# Function 5: Part 2 analysis notebook

This notebook keeps the original data-loading cells, appends the latest query point/output from the end of your uploaded notebook, and then runs one focused analysis for Part 2.

Assumption: lower output is better, so the optimisation target is minimisation.


In [2]:
import numpy as np

input_data = np.load('../../data/initial_data/function_5/initial_inputs.npy')
print("Before:", input_data.shape)
new_point = np.array([
    [0.278167, 0.217734, 0.996929, 0.992772],
    [0.331025, 0.592662, 0.991036, 0.994402],
    [0.5, 0.5, 0.5, 0.5]
    ])
input_data = np.vstack([input_data, new_point])
print("After:", input_data.shape)
print(input_data)


Before: (20, 4)
After: (23, 4)
[[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]
 [0.278167   0.217734   0.996

In [ ]:
output_data = np.load('../../data/initial_data/function_5/initial_outputs.npy')
print("Before:", output_data.shape)
new_output = np.array([
    1552.6699801013826,
    1803.9236591562037,
    -0.015979341188442648
    ])
output_data = np.append(output_data, new_output)
print("After:", output_data.shape)
print(output_data)


Before: (20,)
After: (22,)
[6.44434399e+01 1.83013796e+01 1.12939795e-01 4.21089813e+00
 2.58370525e+02 7.84343889e+01 5.75715369e+01 1.09571876e+02
 8.84799176e+00 2.33223610e+02 2.44230883e+01 6.44201468e+01
 6.34767158e+01 7.97291299e+01 3.55806818e+02 1.08885962e+03
 2.88667516e+01 4.51815703e+01 4.31612757e+02 9.97233189e+00
 1.55266998e+03 1.80392366e+03]


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Latest query/output extracted from the uploaded notebook.
# Edit these if you later receive a different portal output.
latest_query = np.array([[0.5, 0.5, 0.5, 0.5]])
actual_output = 32.0025

# Append the latest query/output only if it is not already present.
if actual_output is not None:
    already_present = np.any(np.all(np.isclose(input_data, latest_query, atol=1e-12), axis=1))
    if not already_present:
        input_data = np.vstack([input_data, latest_query])
        output_data = np.append(output_data, actual_output)

function_id = 5
d = input_data.shape[1]
print(f"Function {function_id}, dimension d={d}")
print("Data shape:", input_data.shape, output_data.shape)
print("Current best observed y:", output_data.min())
print("Current best x:", input_data[np.argmin(output_data)])


Function 5, dimension d=4
Data shape: (23, 4) (23,)
Current best observed y: 0.1129397953712203
Current best x: [0.43834987 0.8043397  0.21024527 0.15129482]


In [4]:
# Basic table used in all interpretations
summary = pd.DataFrame(input_data, columns=[f"x{i+1}" for i in range(d)])
summary["y"] = output_data
summary["log_abs_y"] = np.log(np.abs(output_data) + 1e-300)
summary["rank_min"] = summary["y"].rank(method="first", ascending=True).astype(int)
summary = summary.sort_values("y")
display(summary)


,x1,x2,x3,x4,y,log_abs_y,rank_min
2,0.438350,0.804340,0.210245,0.151295,0.112940,-2.180900,1
3,0.706051,0.534192,0.264243,0.482088,4.210898,1.437676,2
8,0.153786,0.729382,0.422598,0.443074,8.847992,2.180191,3
19,0.126885,0.153430,0.770162,0.190518,9.972332,2.299814,4
1,0.758653,0.536518,0.656000,0.360342,18.301380,2.906976,5
10,0.677491,0.358510,0.479592,0.072880,24.423088,3.195529,6
16,0.725262,0.479870,0.088947,0.759760,28.866752,3.362690,7
22,0.500000,0.500000,0.500000,0.500000,32.002500,3.465814,8
17,0.355482,0.639619,0.417618,0.122604,45.181570,3.810689,9
6,0.553621,0.667350,0.323806,0.814870,57.571537,4.053028,10


## Classification framing: good vs bad outputs

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# Define "good" as the best quartile of observed outputs.
# For a very small dataset this gives enough positive labels to fit a classifier.
good_threshold = np.quantile(output_data, 0.25)
good_label = (output_data <= good_threshold).astype(int)

class_df = pd.DataFrame(input_data, columns=[f"x{i+1}" for i in range(d)])
class_df["y"] = output_data
class_df["good_label"] = good_label
class_df["distance_to_threshold"] = np.abs(output_data - good_threshold)
class_df["log_abs_y"] = np.log(np.abs(output_data) + 1e-300)
class_df = class_df.sort_values("distance_to_threshold")

print(f"Good/bad threshold: y <= {good_threshold:.6e}")
display(class_df)

print("Support-vector-like observed points:")
display(class_df.head(min(5, len(class_df))))


Good/bad threshold: y <= 2.664492e+01


,x1,x2,x3,x4,y,good_label,distance_to_threshold,log_abs_y
10,0.677491,0.358510,0.479592,0.072880,24.423088,1,2.221832,3.195529
16,0.725262,0.479870,0.088947,0.759760,28.866752,0,2.221832,3.362690
22,0.500000,0.500000,0.500000,0.500000,32.002500,0,5.357580,3.465814
1,0.758653,0.536518,0.656000,0.360342,18.301380,1,8.343540,2.906976
19,0.126885,0.153430,0.770162,0.190518,9.972332,1,16.672588,2.299814
8,0.153786,0.729382,0.422598,0.443074,8.847992,1,17.796928,2.180191
17,0.355482,0.639619,0.417618,0.122604,45.181570,0,18.536650,3.810689
3,0.706051,0.534192,0.264243,0.482088,4.210898,1,22.434022,1.437676
2,0.438350,0.804340,0.210245,0.151295,0.112940,1,26.531980,-2.180900
6,0.553621,0.667350,0.323806,0.814870,57.571537,0,30.926617,4.053028


Support-vector-like observed points:


,x1,x2,x3,x4,y,good_label,distance_to_threshold,log_abs_y
10,0.677491,0.358510,0.479592,0.072880,24.423088,1,2.221832,3.195529
16,0.725262,0.479870,0.088947,0.759760,28.866752,0,2.221832,3.362690
22,0.500000,0.500000,0.500000,0.500000,32.002500,0,5.357580,3.465814
1,0.758653,0.536518,0.656000,0.360342,18.301380,1,8.343540,2.906976
19,0.126885,0.153430,0.770162,0.190518,9.972332,1,16.672588,2.299814


In [6]:
X = input_data.copy()
y_cls = good_label
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

if len(np.unique(y_cls)) < 2:
    print("Only one class is present, so logistic/SVM classifiers cannot be fitted yet.")
else:
    log_reg = LogisticRegression(class_weight="balanced", random_state=0)
    svm_linear = SVC(kernel="linear", class_weight="balanced", probability=True, random_state=0)
    svm_rbf = SVC(kernel="rbf", C=10.0, gamma="scale", class_weight="balanced", probability=True, random_state=0)

    models = {"logistic": log_reg, "linear_svm": svm_linear, "rbf_svm": svm_rbf}
    for name, model in models.items():
        model.fit(X_scaled, y_cls)
        pred = model.predict(X_scaled)
        print("\n", name)
        print("Confusion matrix:\n", confusion_matrix(y_cls, pred))
        print(classification_report(y_cls, pred, zero_division=0))



 logistic
Confusion matrix:
 [[14  3]
 [ 0  6]]
              precision    recall  f1-score   support

           0       1.00      0.82      0.90        17
           1       0.67      1.00      0.80         6

    accuracy                           0.87        23
   macro avg       0.83      0.91      0.85        23
weighted avg       0.91      0.87      0.88        23


 linear_svm
Confusion matrix:
 [[13  4]
 [ 0  6]]
              precision    recall  f1-score   support

           0       1.00      0.76      0.87        17
           1       0.60      1.00      0.75         6

    accuracy                           0.83        23
   macro avg       0.80      0.88      0.81        23
weighted avg       0.90      0.83      0.84        23


 rbf_svm
Confusion matrix:
 [[17  0]
 [ 0  6]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        17
           1       1.00      1.00      1.00         6

    accuracy                      

In [7]:
# Boundary / uncertainty search: where classifier is closest to p(good)=0.5.
if len(np.unique(y_cls)) >= 2:
    rng = np.random.default_rng(1)
    candidates = rng.random((30000 if d <= 4 else 60000, d))
    cand_scaled = scaler.transform(candidates)

    rows = []
    for name, model in models.items():
        prob_good = model.predict_proba(cand_scaled)[:, 1]
        uncertainty = np.abs(prob_good - 0.5)
        idx = np.argsort(uncertainty)[:10]
        tmp = pd.DataFrame(candidates[idx], columns=[f"x{i+1}" for i in range(d)])
        tmp["model"] = name
        tmp["p_good"] = prob_good[idx]
        tmp["boundary_score_abs_p_minus_0.5"] = uncertainty[idx]
        rows.append(tmp)

    boundary_points = pd.concat(rows, ignore_index=True)
    display(boundary_points.sort_values("boundary_score_abs_p_minus_0.5").head(20))

    candidate = boundary_points.sort_values("boundary_score_abs_p_minus_0.5").iloc[0][[f"x{i+1}" for i in range(d)]].to_numpy(float)
    print("Most boundary-like candidate:", np.round(candidate, 6))
    print("Portal format:", ", ".join(f"x{i+1}={v:.6f}" for i, v in enumerate(candidate)))


,x1,x2,x3,x4,model,p_good,boundary_score_abs_p_minus_0.5
29,0.800862,0.326117,0.031693,0.012139,rbf_svm,0.500000,0.000000
27,0.978709,0.883288,0.468198,0.255061,rbf_svm,0.500000,0.000000
26,0.142293,0.870121,0.097640,0.692183,rbf_svm,0.500000,0.000000
25,0.782550,0.697305,0.513495,0.474495,rbf_svm,0.500000,0.000000
24,0.650090,0.888613,0.379064,0.361892,rbf_svm,0.500000,0.000000
23,0.576142,0.726628,0.139651,0.444187,rbf_svm,0.500000,0.000000
22,0.330429,0.900465,0.060237,0.066867,rbf_svm,0.500000,0.000000
21,0.910414,0.780109,0.652307,0.183135,rbf_svm,0.500000,0.000000
20,0.388470,0.909238,0.278785,0.422633,rbf_svm,0.500000,0.000000
28,0.280195,0.698708,0.069405,0.398714,rbf_svm,0.500000,0.000000


Most boundary-like candidate: [0.800862 0.326117 0.031693 0.012139]
Portal format: x1=0.800862, x2=0.326117, x3=0.031693, x4=0.012139


In [8]:
# Optional 2D plot
if d == 2 and len(np.unique(y_cls)) >= 2:
    grid_res = 200
    xx, yy = np.meshgrid(np.linspace(0, 1, grid_res), np.linspace(0, 1, grid_res))
    grid = np.c_[xx.ravel(), yy.ravel()]
    grid_scaled = scaler.transform(grid)
    for name, model in models.items():
        prob = model.predict_proba(grid_scaled)[:, 1].reshape(grid_res, grid_res)
        plt.figure(figsize=(7, 6))
        cf = plt.contourf(xx, yy, prob, levels=40)
        plt.colorbar(cf, label="Predicted P(good)")
        plt.contour(xx, yy, prob, levels=[0.5], linewidths=2)
        plt.scatter(input_data[:, 0], input_data[:, 1], c=good_label, edgecolors="black", s=80)
        plt.xlabel("x1"); plt.ylabel("x2"); plt.title(f"Good/bad boundary: {name}")
        plt.show()
